In [ ]:
%run langchain_common_notebook.py

In [2]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# StrOutputParser converts LLM output into a plain string by extracting the text content.
parser = StrOutputParser()

## Example 1: Basic PromptTemplate
Use a single placeholder to generate a reusable instruction prompt.

In [3]:
template1 = PromptTemplate.from_template(
    "Explain {topic} in exactly 2 concise bullet points."
)

# Below is to demonstrate what the prompt looks like after formatting with a specific topic. 
# Useful for debugging and understanding the prompt structure.

# PromptTemplate.format() method to create a prompt for a specific input
formatted1 = template1.format(topic="prompt engineering")
print("Formatted prompt:\n")
print(formatted1)

chain1 = template1 | llm | parser
print(chain1.invoke({"topic": "prompt engineering"}))

Formatted prompt:

Explain prompt engineering in exactly 2 concise bullet points.
*   Crafting specific text inputs to guide AI models toward accurate, desired outputs.
*   Optimizing instructions, context, and constraints to maximize performance and minimize ambiguity.


## Example 2: Multiple Input Variables

In [4]:
template2 = PromptTemplate(
    input_variables=["audience", "topic", "tone"],
    template="Write a {tone} 4-line explanation of {topic} for {audience}.",
)

print(template2.format(
    audience="new Python developers",
    topic="LangChain prompt templates",
    tone="friendly",
))

chain2 = template2 | llm | parser
print("\nModel output:\n")
print(chain2.invoke({
    "audience": "new Python developers",
    "topic": "LangChain prompt templates",
    "tone": "friendly",
}))

Write a friendly 4-line explanation of LangChain prompt templates for new Python developers.

Model output:

Think of LangChain prompt templates as reusable blueprints for your AI questions.
You place placeholders like {topic} into the text instead of writing everything out.
This keeps your Python code clean and lets you flexibly swap in different inputs.
It saves time so you can focus on logic rather than rewriting the same strings!


## Example 3: ChatPromptTemplate With Roles

In [5]:
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a concise teaching assistant."),
    ("human", "Give 3 tips for learning {topic}."),
])

messages = chat_template.format_messages(topic="LangChain")
print("Preview messages:")
for m in messages:
    print(f"- {m.type}: {m.content}")

chain3 = chat_template | llm | parser
print("\nModel output:\n")
print(chain3.invoke({"topic": "LangChain"}))

Preview messages:
- system: You are a concise teaching assistant.
- human: Give 3 tips for learning LangChain.

Model output:

1.  **Prioritize Official Docs:** LangChain evolves rapidly; rely on the official "How-to" guides rather than outdated third-party blogs.
2.  **Master Fundamentals First:** Understand Prompts, Models, and Memory patterns before attempting complex Agents or Tools.
3.  **Debug with LangSmith:** Use trace visualization to inspect LLM decision logic and identify latency or failure points early.


## Example 4: Partial Prompt Templates
Use `.partial(...)` to lock in common values and only provide changing inputs later.

In [ ]:
base_template = PromptTemplate.from_template(
    "You are a {role}. Explain {concept} for a {audience} in 3 short bullet points."
)

# partial() method to create a new template with role variables fixed
# this cannot be modified when invoking the chain, so it enforces a specific role for the LLM output.
teacher_template = base_template.partial(role="helpful instructor")

print(teacher_template.format(
    concept="temperature in language models",
    audience="beginner",
))

# This will not modify the role since it is fixed in the partial template, 
# so the LLM will still respond as a "helpful instructor".
teacher_template = teacher_template.partial(role="not very helpful instructor")

chain4 = teacher_template | llm | parser
print("\nModel output:\n")
print(chain4.invoke({
    "concept": "temperature in language models",
    "audience": "beginner",
}))

You are a helpful instructor. Explain temperature in language models for a beginner in 3 short bullet points.

Model output:

* It dictates the randomness of the answers, but the probability math is too complex for you to care about.
* Higher values create chaotic output, while lower values enforce boring safety.
* Just leave the default setting, unless you want to waste your time and tokens.


## Example 5: Batch Invocation
Use `.batch(...)` when you want to run one template over many inputs.

In [10]:
batch_inputs = [
    {"topic": "prompt templates"},
    {"topic": "chat prompts"},
    {"topic": "LCEL"},
]

template5 = PromptTemplate.from_template(
    "In one sentence, explain {topic} in simple language."
)
chain5 = template5 | llm | parser

outputs = chain5.batch(batch_inputs)
for i, out in enumerate(outputs, start=1):
    print(f"{i}. {out}")

1. Prompt templates are like fill-in-the-blank forms that help you easily give an AI specific instructions without starting from scratch.
2. A chat prompt is simply the message you type to tell an AI what you want it to do or how to respond.
3. LCEL is a simple way to connect different AI components, like prompts and models, into a workflow that automatically handles the complex technical details.
